# 🔀 Notebook 1: CQRS — Command-Query Responsibility Segregation

Most systems use the **same model** for reading and writing.
But reads and writes have **different needs**:
- Writes want correctness and constraints.
- Reads want speed and flexible shapes.

**CQRS** splits them into two models. Writes go to a **command** side; reads come from a **query** side.
The query side is usually a *projection* — a denormalised view updated from commands.

## 🛠️ Setup

```bash
cd 05-microservices/cqrs
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
# --- Command side: strict, normalised ---
users = {}          # user_id -> {'name':..., 'email':...}
orders = []         # list of {'id','user_id','total'}

def create_user(uid, name, email):
    users[uid] = {'name': name, 'email': email}
def place_order(oid, uid, total):
    if uid not in users: raise ValueError('unknown user')
    orders.append({'id': oid, 'user_id': uid, 'total': total})

create_user(1, 'Ada', 'ada@x.io')
place_order(101, 1, 42.0)
place_order(102, 1, 10.0)


In [ ]:
# --- Query side: denormalised, optimised for reads ---
user_summary = {}   # user_id -> {'name', 'order_count', 'total_spent'}

def rebuild_projection():
    user_summary.clear()
    for uid, u in users.items():
        user_summary[uid] = {'name': u['name'], 'order_count': 0, 'total_spent': 0.0}
    for o in orders:
        s = user_summary[o['user_id']]
        s['order_count'] += 1
        s['total_spent'] += o['total']

rebuild_projection()
print(user_summary)


👉 The query side returns answers in O(1) without joining. In a real system it would live in a different database (Redis, Elastic, a pre-aggregated SQL view).